# Train the Level 1 action classifier on the Google Drive dataset

Runs `app/train_actions.py` (branch `feat/action-classifier`) on the full dataset in
`MyDrive/PatternRecogSystemDataSet`:

- one sample per logged action (move, click, type, press_key, hotkey, drag), features from the **video only**
- **70 / 20 / 10** train / validation / test split **by recording**, stratified by app
- Tier 1 (SVM, Naive Bayes, Decision Tree, Random Forest, KNN, XGBoost, LightGBM) + MLP;
  each setting is chosen on validation macro F1 and scored **once** on test

**Runtime:** a CPU runtime is enough (the work is video decoding, not deep learning).
Outputs (models, `labels.json`, `split.json`, `metrics.json`, `results.md`) are copied back to Drive.

Run the cells in order. Start with `LIMIT_PER_APP = 5` for a quick trial, then set it to `0` for the full run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Settings

In [ ]:
DATASET_DRIVE = '/content/drive/MyDrive/PatternRecogSystemDataSet'          # the dataset folder on Drive
OUTPUT_DRIVE  = '/content/drive/MyDrive/PatternRecogSystemModels/actions/v1' # results are copied here
BRANCH        = 'feat/action-classifier'
GRANULARITY   = 'fine'      # 'fine' (6 classes, for SOP steps) or 'coarse' (proposal's 4)
MODELS        = 'svm,naive_bayes,decision_tree,random_forest,knn,xgboost,lightgbm,mlp'
LIMIT_PER_APP = 5           # 5 = quick trial; 0 = full dataset

LOCAL_DATASET = '/content/dataset'
LOCAL_OUTPUT  = '/content/out/actions'

## 2. Look at the dataset

The training code expects one folder per recording containing `action_log.json` and
`video/video.mp4` (any nesting above that is fine). Zip files are extracted in the next step.

In [ ]:
import pathlib, collections
root = pathlib.Path(DATASET_DRIVE)
assert root.exists(), f'{DATASET_DRIVE} not found - check the path in Settings'
zips = sorted(root.rglob('*.zip'))
logs = sorted(root.rglob('action_log.json'))
videos = sorted(root.rglob('video.mp4'))
print(f'zip files: {len(zips)}   action_log.json: {len(logs)}   video.mp4: {len(videos)}')
print('top-level entries:', sorted(p.name for p in root.iterdir())[:20])
for p in logs[:3]:
    print(' e.g.', p.relative_to(root))

## 3. Copy to the Colab disk

Reading video frames straight from Drive is very slow, so copy (or unzip) the dataset to local disk first.
With `LIMIT_PER_APP` set, everything is still copied; the limit applies when training.

In [ ]:
import shutil, subprocess, zipfile
dst = pathlib.Path(LOCAL_DATASET)
dst.mkdir(parents=True, exist_ok=True)
for z in zips:
    target = dst / z.stem
    if not target.exists():
        print('unzipping', z.name)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(target)
subprocess.run(['rsync', '-a', '--exclude', '*.zip', f'{root}/', f'{dst}/'], check=True)
print(subprocess.run(['du', '-sh', str(dst)], capture_output=True, text=True).stdout)
print('action logs on local disk:', len(list(dst.rglob('action_log.json'))))

## 4. Get the code and dependencies

In [ ]:
%cd /content
!rm -rf PatternRecognSystem-Video2Text
!git clone --depth 1 -b {BRANCH} https://github.com/laxmi1707/PatternRecognSystem-Video2Text.git
!pip install -q xgboost lightgbm opencv-python-headless scikit-learn
%cd /content/PatternRecognSystem-Video2Text/SourceCode/backend
!git log --oneline -1

## 5. Train

Feature extraction is the slow part (it reads every video once); it is cached in
`features.npz`, so retraining with other models can use `--load-features`.

In [ ]:
limit = f'--limit-per-app {LIMIT_PER_APP}' if LIMIT_PER_APP else ''
!python -m app.train_actions --dataset-root {LOCAL_DATASET} --model-dir {LOCAL_OUTPUT}     --granularity {GRANULARITY} --models {MODELS} {limit}

## 6. Save to Drive and show the results

In [ ]:
out = pathlib.Path(OUTPUT_DRIVE)
out.mkdir(parents=True, exist_ok=True)
shutil.copytree(LOCAL_OUTPUT, out, dirs_exist_ok=True)
print('saved to', out, ':', sorted(p.name for p in out.iterdir()))
from IPython.display import Markdown, display
display(Markdown((out / 'results.md').read_text()))

## Optional: retrain without re-reading the videos

For example after changing a model's settings grid. The cache is tied to the granularity it was built with.

In [ ]:
# !python -m app.train_actions --model-dir {LOCAL_OUTPUT} --load-features --models xgboost,lightgbm